# Predicting food inflation in Turkey

**Question:** Do exchange rates and global food prices help predict monthly food inflation in Turkey?

We will compare Turkish food-price changes with USD/TRY changes and global food-price changes, including related food groups.

The cell below downloads three CSV files into `data/raw/`: `turkish_food_cpi.csv`, `usd_try_daily.csv` and `fao_food_prices.csv`. Turkish CPI and USD/TRY are requested from January 2005 through the last completed month. Yearly EVDS responses are combined before saving each source. Each source's original column names and date formats are retained. No monthly aggregation, gap filling or merging between sources is done.

- **Turkish food CPI and food categories:** [TÜİK](https://veriportali.tuik.gov.tr/tr/press/58289), downloaded through [TCMB EVDS](https://evds3.tcmb.gov.tr/). CPI is a price index; 2025=100 is its reference scale.
- **USD/TRY exchange rate:** [TCMB EVDS](https://evds3.tcmb.gov.tr/), daily buying rates.
- **Global food prices:** [FAO Food Price Index](https://www.fao.org/worldfoodsituation/foodpricesindex/en/), monthly overall and food-group indices. The CSV includes its full available history.

Install the libraries listed in `requirements.txt` in the notebook's Python environment with `python -m pip install -r requirements.txt`. To get the free API key, open [EVDS](https://evds3.tcmb.gov.tr/), choose “Benim Sayfam / My Page”, register if needed, and sign in. Open your username menu, choose “Profilim / Profile”, and select “API Anahtarını Kopyala / Copy API Key” near the bottom. Paste the key after `EVDS_API_KEY=` in the local `.env` file and save it. The cell reads that file automatically and uses a hidden prompt if the key is still empty. The `.env` file is excluded from Git. Do not paste the key into the notebook code or this chat. See the [official API guide](https://evds3.tcmb.gov.tr/igmevdsms-dis/documents/showDocument?docId=16). The EVDS requests are split by year to stay below the API's 1,000-observation limit. FAO's title rows and unused trailing columns are omitted from the saved CSV.

TCMB's [FAQ](https://www.tcmb.gov.tr/wps/wcm/connect/bab69efb-d66c-45b5-91c6-1533886acd6e/GenelAg-SSS.pdf?CACHE=NONE&CONTENTCACHE=NONE&MOD=AJPERES&attachment=true) describes EVDS exchange rates as rates determined on the previous business day. Our spot checks match that convention. We retain the EVDS dates rather than shifting them.


In [6]:
import io
import os
from datetime import date, timedelta
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
end_date = date.today().replace(day=1) - timedelta(days=1)
load_dotenv(".env")
api_key = os.getenv("EVDS_API_KEY", "").strip() or getpass("EVDS API key: ").strip()
if not api_key:
    raise ValueError("Enter your EVDS API key in .env or the hidden prompt.")

fao_url = (
    "https://www.fao.org/media/docs/worldfoodsituationlibraries/wfs-library/food_price_indices_data.csv?download=true&sfvrsn=523ebd2a_84"
)
response = requests.get(fao_url, timeout=90)
response.raise_for_status()
fao = pd.read_csv(io.BytesIO(response.content), skiprows=2, usecols=range(7)).dropna(how="all")
fao.to_csv(raw_dir / "fao_food_prices.csv", index=False)
print(f"Saved fao_food_prices.csv: {len(fao)} months")

series = {
    "turkish_food_cpi": [
        "TP.TUKFIY2025.01",
        "TP.TUKFIY2025.0111",
        "TP.TUKFIY2025.0112",
        "TP.TUKFIY2025.0114",
        "TP.TUKFIY2025.0115",
    ],
    "usd_try_daily": ["TP.DK.USD.A.YTL"],
}
for dataset, codes in series.items():
    yearly_tables = []
    for year in range(2005, end_date.year + 1):
        year_end = min(date(year, 12, 31), end_date)
        url = (
            f"https://evds3.tcmb.gov.tr/igmevdsms-dis/series={'-'.join(codes)}&startDate=01-01-{year}&endDate={year_end:%d-%m-%Y}&type=csv"
        )
        response = requests.get(url, headers={"key": api_key}, timeout=90)
        response.raise_for_status()
        yearly_tables.append(pd.read_csv(io.BytesIO(response.content)))

    data = pd.concat(yearly_tables, ignore_index=True)
    data.to_csv(raw_dir / f"{dataset}.csv", index=False)
    print(f"Saved {dataset}.csv: {len(data)} observations")


Saved fao_food_prices.csv: 441 months
Saved turkish_food_cpi.csv: 261 observations
Saved usd_try_daily.csv: 7943 observations


In [12]:
import pandas as pd

cpi = pd.read_csv("data/raw/turkish_food_cpi.csv").rename(columns={
    "Tarih": "date",
    "TP_TUKFIY2025_01": "food_cpi",
    "TP_TUKFIY2025_0111": "cereals_cpi",
    "TP_TUKFIY2025_0112": "meat_cpi",
    "TP_TUKFIY2025_0114": "dairy_eggs_cpi",
    "TP_TUKFIY2025_0115": "oils_fats_cpi",
    "UNIXTIME": "unix_time",
})

usd_try = pd.read_csv("data/raw/usd_try_daily.csv").rename(columns={
    "Tarih": "date",
    "TP_DK_USD_A_YTL": "usd_try",
    "UNIXTIME": "unix_time",
})

fao = pd.read_csv("data/raw/fao_food_prices.csv").rename(columns={
    "Date": "date",
    "Food Price Index": "food_price_index",
    "Meat": "meat",
    "Dairy": "dairy",
    "Cereals": "cereals",
    "Oils": "vegetable_oils",
    "Sugar": "sugar",
})


In [22]:
print("shape:", cpi.shape)
print("-------------------------")

print("info")
cpi.info()
print("-------------------------")

print("describe")
print(cpi.describe())

print("-------------------------")

print("head 5")
cpi.head()

shape: (261, 7)
-------------------------
info
<class 'pandas.DataFrame'>
RangeIndex: 261 entries, 0 to 260
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            261 non-null    str    
 1   food_cpi        261 non-null    float64
 2   cereals_cpi     261 non-null    float64
 3   meat_cpi        261 non-null    float64
 4   dairy_eggs_cpi  261 non-null    float64
 5   oils_fats_cpi   261 non-null    float64
 6   unix_time       261 non-null    str    
dtypes: float64(5), str(2)
memory usage: 14.4 KB
-------------------------
describe
         food_cpi  cereals_cpi    meat_cpi  dairy_eggs_cpi  oils_fats_cpi
count  261.000000   261.000000  261.000000      261.000000     261.000000
mean    21.089693    20.869004   20.641724       22.096513      22.515862
std     32.458199    31.560159   33.746475       32.569547      32.686775
min      2.430000     2.480000    2.010000        2.830000       2.840000
2

,date,food_cpi,cereals_cpi,meat_cpi,dairy_eggs_cpi,oils_fats_cpi,unix_time
0,2005-1,2.50,2.48,2.01,3.07,2.88,"{""numberLong"":""1104534000""}"
1,2005-2,2.52,2.49,2.04,3.08,2.89,"{""numberLong"":""1107212400""}"
2,2005-3,2.52,2.49,2.06,3.07,2.89,"{""numberLong"":""1109631600""}"
3,2005-4,2.50,2.49,2.15,3.03,2.87,"{""numberLong"":""1112306400""}"
4,2005-5,2.50,2.49,2.16,2.96,2.86,"{""numberLong"":""1114898400""}"


In [15]:
usd_try.head()

,date,usd_try,unix_time
0,01-01-2005,NaN,"{""numberLong"":""1104534000""}"
1,02-01-2005,NaN,"{""numberLong"":""1104620400""}"
2,03-01-2005,1.3363,"{""numberLong"":""1104706800""}"
3,04-01-2005,1.3383,"{""numberLong"":""1104793200""}"
4,05-01-2005,1.3427,"{""numberLong"":""1104879600""}"


In [16]:
fao.head()

,date,food_price_index,meat,dairy,cereals,vegetable_oils,sugar
0,1990-01,64.4,74.3,53.5,64.1,44.59,87.9
1,1990-02,64.7,76.8,52.2,62.2,44.50,90.7
2,1990-03,64.0,78.5,41.4,61.3,45.75,95.1
3,1990-04,66.0,81.2,48.4,62.8,44.02,94.3
4,1990-05,64.6,81.8,39.2,62.0,45.50,90.4
